## SQL - Structured Query Language

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")  # vaqtinchalik baza, RAMda

def q(query):
    return pd.read_sql_query(query, conn)

In [5]:
conn.executescript("""
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT,
    city TEXT,
    signup_date TEXT
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product TEXT,
    amount REAL,
    order_date TEXT,
    FOREIGN KEY(customer_id) REFERENCES customers(id)
);

INSERT INTO customers VALUES
(1, 'Aziz', 'Toshkent', '2024-01-15'),
(2, 'Malika', 'Samarqand', '2024-02-20'),
(3, 'Bekzod', 'Toshkent', NULL),
(4, 'Nodira', 'Buxoro', '2024-03-05'),
(5, 'Javlon', NULL, '2024-01-30');

INSERT INTO orders VALUES
(1, 1, 'Noutbuk', 1200.00, '2024-04-01'),
(2, 1, 'Sichqoncha', 15.50, '2024-04-03'),
(3, 2, 'Klaviatura', 45.00, '2024-04-10'),
(4, 3, 'Monitor', 300.00, '2024-04-12'),
(5, 4, 'Noutbuk', 1100.00, '2024-04-15'),
(6, 2, 'Naushnik', 60.00, '2024-04-18'),
(7, NULL, 'Sim kabel', 5.00, '2024-04-20');
""")
conn.commit()

In [10]:
q("SELECT * FROM customers")

,id,name,city,signup_date
0,1,Aziz,Toshkent,2024-01-15
1,2,Malika,Samarqand,2024-02-20
2,3,Bekzod,Toshkent,NaN
3,4,Nodira,Buxoro,2024-03-05
4,5,Javlon,NaN,2024-01-30


In [8]:
# Aniq ustunlar
q("SELECT name, city FROM customers")


,name,city
0,Aziz,Toshkent
1,Malika,Samarqand
2,Bekzod,Toshkent
3,Nodira,Buxoro
4,Javlon,NaN


In [9]:
# Filtrlash
q("SELECT * FROM customers WHERE city = 'Toshkent'")

,id,name,city,signup_date
0,1,Aziz,Toshkent,2024-01-15
1,3,Bekzod,Toshkent,NaN


In [11]:
# Bir nechta shart
q("SELECT * FROM orders WHERE amount > 100 AND order_date > '2024-04-10'")


,id,customer_id,product,amount,order_date
0,4,3,Monitor,300.0,2024-04-12
1,5,4,Noutbuk,1100.0,2024-04-15


In [12]:
# Tartiblash va cheklash
q("SELECT * FROM orders ORDER BY amount DESC LIMIT 3")

,id,customer_id,product,amount,order_date
0,1,1,Noutbuk,1200.0,2024-04-01
1,5,4,Noutbuk,1100.0,2024-04-15
2,4,3,Monitor,300.0,2024-04-12


In [14]:
q("SELECT * FROM customers WHERE signup_date IS NULL")

,id,name,city,signup_date
0,3,Bekzod,Toshkent,None


In [17]:
q("SELECT * FROM customers")

,id,name,city,signup_date
0,1,Aziz,Toshkent,2024-01-15
1,2,Malika,Samarqand,2024-02-20
2,3,Bekzod,Toshkent,NaN
3,4,Nodira,Buxoro,2024-03-05
4,5,Javlon,NaN,2024-01-30


In [18]:
q("SELECT * FROM orders")

,id,customer_id,product,amount,order_date
0,1,1.0,Noutbuk,1200.0,2024-04-01
1,2,1.0,Sichqoncha,15.5,2024-04-03
2,3,2.0,Klaviatura,45.0,2024-04-10
3,4,3.0,Monitor,300.0,2024-04-12
4,5,4.0,Noutbuk,1100.0,2024-04-15
5,6,2.0,Naushnik,60.0,2024-04-18
6,7,NaN,Sim kabel,5.0,2024-04-20


In [21]:
# # INNER JOIN — faqat ikkalasida ham mos yozuv bo'lsa
# q("""
# SELECT c.name, o.product, o.amount
# FROM customers c
# JOIN orders o ON c.id = o.customer_id
# """)

# LEFT JOIN — chapdagi barcha yozuvlar, mos kelmasa ham
q("""
SELECT c.name, o.product, o.amount
FROM customers c
RIGHT JOIN orders o ON c.id = o.customer_id
""")

,name,product,amount
0,Aziz,Noutbuk,1200.0
1,Aziz,Sichqoncha,15.5
2,Malika,Klaviatura,45.0
3,Malika,Naushnik,60.0
4,Bekzod,Monitor,300.0
5,Nodira,Noutbuk,1100.0
6,NaN,Sim kabel,5.0


## GROUP BY, HAVING

In [22]:
# Har bir mijoz nechta buyurtma bergan
q("""
SELECT customer_id, COUNT(*) as order_count, SUM(amount) as total_spent
FROM orders
GROUP BY customer_id
""")

,customer_id,order_count,total_spent
0,NaN,1,5.0
1,1.0,2,1215.5
2,2.0,2,105.0
3,3.0,1,300.0
4,4.0,1,1100.0


In [23]:
# Shahar bo'yicha o'rtacha
q("""
SELECT c.city, COUNT(o.id) as orders, ROUND(AVG(o.amount), 2) as avg_amount
FROM customers c
JOIN orders o ON c.id = o.customer_id
GROUP BY c.city
""")

,city,orders,avg_amount
0,Buxoro,1,1100.00
1,Samarqand,2,52.50
2,Toshkent,3,505.17


In [24]:
q("""
SELECT c.name, SUM(o.amount) as total
FROM customers c
JOIN orders o ON c.id = o.customer_id
GROUP BY c.name
HAVING total > 100
ORDER BY total DESC
""")

,name,total
0,Aziz,1215.5
1,Nodira,1100.0
2,Bekzod,300.0
3,Malika,105.0


In [ ]:
q("""
SELECT name,
       COALESCE(city, 'Noma''lum') AS city
FROM customers
""")

,name,city
0,Aziz,Toshkent
1,Malika,Samarqand
2,Bekzod,Toshkent
3,Nodira,Buxoro
4,Javlon,Noma'lum


In [29]:
# Sana formatlarini ishlash (SQLite'da date funksiyalari)
q("""
SELECT product, order_date, strftime('%Y-%m', order_date) as month
FROM orders
""")

,product,order_date,month
0,Noutbuk,2024-04-01,2024-04
1,Sichqoncha,2024-04-03,2024-04
2,Klaviatura,2024-04-10,2024-04
3,Monitor,2024-04-12,2024-04
4,Noutbuk,2024-04-15,2024-04
5,Naushnik,2024-04-18,2024-04
6,Sim kabel,2024-04-20,2024-04


In [30]:
q("""
SELECT product, COUNT(*) as cnt
FROM orders
GROUP BY product
HAVING cnt > 1
""")

,product,cnt
0,Noutbuk,2


# Mashq 1: Toshkentdagi barcha mijozlarning ismi va shahrini chiqaring

# Mashq 2: Har bir mahsulot bo'yicha jami sotilgan summa (mahsulot nomi bo'yicha guruhlab)

# Mashq 3: 1000 dan ko'p sarflagan mijozlarning ismini toping (JOIN + GROUP BY + HAVING)

# Mashq 4: Hech qanday buyurtma bermagan mijozlarni toping (LEFT JOIN + WHERE ... IS NULL)

# Mashq 5: Har oyda nechta buyurtma tushganini hisoblang (strftime + GROUP BY)